# Desafio: Analisando Texto sobre Ciência de Dados

Neste exemplo, vamos fazer um exercício simples que cobre todas as etapas de um processo tradicional de ciência de dados. Você não precisa escrever nenhum código, pode apenas clicar nas células abaixo para executá-las e observar o resultado. Como desafio, você é incentivado a testar este código com dados diferentes.

## Objetivo

Nesta lição, temos discutido diferentes conceitos relacionados à Ciência de Dados. Vamos tentar descobrir mais conceitos relacionados fazendo uma **mineração de texto**. Começaremos com um texto sobre Ciência de Dados, extrairemos palavras-chave dele e, em seguida, tentaremos visualizar o resultado.

Como texto, vou usar a página sobre Ciência de Dados da Wikipedia:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Passo 1: Obtendo os Dados

O primeiro passo em todo processo de ciência de dados é obter os dados. Usaremos a biblioteca `requests` para fazer isso:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Passo 2: Transformando os Dados

O próximo passo é converter os dados para a forma adequada para processamento. No nosso caso, baixamos o código-fonte HTML da página, e precisamos convertê-lo em texto simples.

Existem muitas maneiras de fazer isso. Usaremos o [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), uma biblioteca Python popular para analisar HTML. O BeautifulSoup nos permite direcionar elementos HTML específicos, assim podemos focar no conteúdo principal do artigo da Wikipedia e reduzir alguns menus de navegação, barras laterais, rodapés e outros conteúdos irrelevantes (embora algum texto padrão possa ainda permanecer).


Primeiro, precisamos instalar a biblioteca BeautifulSoup para análise de HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Passo 3: Obtendo Insights

O passo mais importante é transformar nossos dados em alguma forma da qual possamos extrair insights. No nosso caso, queremos extrair palavras-chave do texto e ver quais palavras-chave são mais significativas.

Usaremos uma biblioteca Python chamada [RAKE](https://github.com/aneesha/RAKE) para extração de palavras-chave. Primeiro, vamos instalar esta biblioteca caso ela não esteja presente: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

A funcionalidade principal está disponível a partir do objeto `Rake`, que podemos personalizar usando alguns parâmetros. No nosso caso, definiremos o comprimento mínimo de uma palavra-chave para 5 caracteres, a frequência mínima de uma palavra-chave no documento para 3 e o número máximo de palavras em uma palavra-chave - para 2. Sinta-se à vontade para experimentar outros valores e observar o resultado.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Obtivemos uma lista de termos junto com o grau associado de importância. Como você pode ver, as disciplinas mais relevantes, como aprendizado de máquina e big data, estão presentes na lista nas posições superiores.

## Passo 4: Visualizando o Resultado

As pessoas conseguem interpretar os dados melhor em forma visual. Portanto, muitas vezes faz sentido visualizar os dados para extrair alguns insights. Podemos usar a biblioteca `matplotlib` em Python para traçar uma distribuição simples das palavras-chave com sua relevância:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Há, no entanto, uma maneira ainda melhor de visualizar as frequências de palavras - usando **Nuvem de Palavras**. Precisaremos instalar outra biblioteca para plotar a nuvem de palavras a partir da nossa lista de palavras-chave.


In [ ]:
!{sys.executable} -m pip install wordcloud

O objeto `WordCloud` é responsável por receber tanto o texto original quanto uma lista pré-calculada de palavras com suas frequências, e retorna uma imagem, que pode então ser exibida usando o `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Também podemos passar o texto original para `WordCloud` - vamos ver se conseguimos obter um resultado semelhante:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Você pode ver que a nuvem de palavras agora parece mais impressionante, mas também contém muito ruído (por exemplo, palavras não relacionadas como `Retrieved on`). Além disso, obtemos menos palavras-chave que consistem em duas palavras, como *data scientist* ou *computer science*. Isso ocorre porque o algoritmo RAKE faz um trabalho muito melhor na seleção de boas palavras-chave a partir do texto. Este exemplo ilustra a importância do pré-processamento e limpeza dos dados, pois uma imagem clara no final nos permitirá tomar decisões melhores.

Neste exercício, passamos por um processo simples de extrair algum significado do texto da Wikipedia, na forma de palavras-chave e nuvem de palavras. Este exemplo é bastante simples, mas demonstra bem todas as etapas típicas que um cientista de dados fará ao trabalhar com dados, começando desde a aquisição dos dados até a visualização.

No nosso curso, discutiremos todas essas etapas em detalhes.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
